# TRACER — Module 6: Attention Analysis & Heatmap Generation

Generates real forensic heatmaps — both gradient saliency and attention visualization — on a
real image and its real PGD-attacked version, using the pipeline from Modules 3-5.

Self-contained — clones the repo, logs into Hugging Face (avoids the rate-limit stalls from
Module 5), and copies Flickr8k to local disk automatically.

In [ ]:
GITHUB_USERNAME = "IqraYasmin123"   # change if this isn't your username
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/tracers.git"

import os
if not os.path.exists("/content/tracers"):
    !git clone {REPO_URL} /content/tracers
else:
    %cd /content/tracers
    !git pull

%cd /content/tracers/ai-engine
import sys
sys.path.insert(0, ".")

from src.vlm import CLIPVLM, VLMConfig
from src.dataset import FlickrDataset, DatasetConfig
from src.attacks import PGDAttack, AttackConfig
from src.attribution import GradientSaliency, AttentionMap, show_attribution
import torch
print("Imports OK.")


## Log into Hugging Face (recommended)

Avoids the rate-limit stalls seen in Module 5. Get a free token from
https://huggingface.co/settings/tokens if you don't have one yet.

In [ ]:
from huggingface_hub import login
HF_TOKEN = ""  # paste your token here, e.g. "hf_..."
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("No token provided — continuing without login (may hit rate limits).")


## Load CLIP and Flickr8k (local-disk copy, same speed fix as Module 5)

The copy is verified complete via a marker file, not just folder existence — if a previous
copy was interrupted, this correctly redoes it rather than silently using partial data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

vlm = CLIPVLM(VLMConfig())
print(f"CLIP loaded on {vlm.device}")

DRIVE_DATASET_DIR = "/content/drive/MyDrive/TRACER/datasets/flickr8k"
LOCAL_DATASET_DIR = "/content/flickr8k_local"
COPY_MARKER = os.path.join(LOCAL_DATASET_DIR, ".copy_complete")

if not os.path.exists(COPY_MARKER):
    print("Copying Flickr8k to local disk (redoing from scratch if a previous attempt was "
          "interrupted)...")
    !rm -rf "{LOCAL_DATASET_DIR}"
    !cp -r "{DRIVE_DATASET_DIR}" "{LOCAL_DATASET_DIR}"
    with open(COPY_MARKER, "w") as f:
        f.write("done")
    print("Copy complete.")
else:
    print("Local copy already present and verified complete.")

dataset = FlickrDataset(DatasetConfig(root_dir=LOCAL_DATASET_DIR, image_size=(224, 224)))
print(f"Loaded {len(dataset)} (image, caption) pairs.")


## Pick a sample and attack it with PGD

In [ ]:
sample = dataset[0]
real_image = sample["image"]
true_caption = sample["caption"]

pixel_values = vlm.preprocess_image(real_image)
text_embeds = vlm.encode_text([true_caption])

pgd_adv = PGDAttack(AttackConfig(epsilon=8/255, alpha=2/255, steps=10)).generate(
    vlm, pixel_values, text_embeds
)

clean_sim = vlm.similarity(vlm.encode_image(pixel_values), text_embeds).item()
adv_sim = vlm.similarity(vlm.encode_image(pgd_adv), text_embeds).item()
print(f"Caption: {true_caption}")
print(f"Clean similarity: {clean_sim:.4f}")
print(f"PGD adversarial similarity: {adv_sim:.4f}")


## Gradient saliency: clean vs. adversarial

Shows which pixels the similarity score is most sensitive to, for both the clean and
attacked image — comparing the two heatmaps is the actual forensic question: did the attack
concentrate its damage in a specific region?

In [ ]:
def to_pil(pixel_values, processor, device):
    mean = torch.tensor(processor.image_processor.image_mean).view(1, 3, 1, 1).to(device)
    std = torch.tensor(processor.image_processor.image_std).view(1, 3, 1, 1).to(device)
    img = (pixel_values * std + mean).clamp(0, 1)
    arr = (img.squeeze().permute(1, 2, 0).detach().cpu().numpy() * 255).astype("uint8")
    from PIL import Image
    return Image.fromarray(arr)

clean_pil = to_pil(pixel_values, vlm.processor, vlm.device)
adv_pil = to_pil(pgd_adv, vlm.processor, vlm.device)

clean_saliency = GradientSaliency().generate(vlm, pixel_values, text_embeds)
adv_saliency = GradientSaliency().generate(vlm, pgd_adv, text_embeds)

fig1 = show_attribution(clean_pil, clean_saliency)


In [ ]:
fig2 = show_attribution(adv_pil, adv_saliency)


## Attention visualization: what is CLIP actually looking at?

In [ ]:
clean_attention = AttentionMap(layer_index=-1).generate(vlm, pixel_values)
fig3 = show_attribution(clean_pil, clean_attention)


In [ ]:
adv_attention = AttentionMap(layer_index=-1).generate(vlm, pgd_adv)
fig4 = show_attribution(adv_pil, adv_attention)


## Module 6 completion check

Run the full test suite:
```bash
cd ai-engine
pytest tests/test_attribution.py -v
```
All 10 tests should pass. Combined with the real heatmaps above — compare the clean vs.
adversarial gradient saliency maps, and the clean vs. adversarial attention maps, and note
whether the attack visibly shifted either signal — Module 6 is complete. Continue to
**Module 7 — Image Reconstruction**.